In [1]:
import os
import tarfile
import random
import re
import librosa
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
from torchaudio.utils import _download_asset
from torch.utils.data import Dataset, DataLoader
from jiwer import wer
from tqdm.auto import tqdm
import whisper

c:\Users\itism\Desktop\test\ml-asr\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)

In [3]:
class Config:
    sr = 16000
    batch_size = 4
    epochs = 10
    lr = 1e-3
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tar_path = "../data/raw/ru_train_0.tar"
    tsv_path = "../data/raw/train(1).tsv"
    extract_dir = "../data/raw/ru_train_data"
    
    duration_sec = 3.0
    target_samples = int(sr * duration_sec)

In [4]:
class ComplexConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding=0):
        super().__init__()
        self.conv_r = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)
        self.conv_i = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)

    def forward(self, x_r, x_i):
        out_r = self.conv_r(x_r) - self.conv_i(x_i)
        out_i = self.conv_r(x_i) + self.conv_i(x_r)
        return out_r, out_i

class ComplexConvTranspose2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding, output_padding):
        super().__init__()
        self.conv_t_r = nn.ConvTranspose2d(in_channels, out_channels, kernel_size, stride, padding, output_padding)
        self.conv_t_i = nn.ConvTranspose2d(in_channels, out_channels, kernel_size, stride, padding, output_padding)

    def forward(self, x_r, x_i):
        out_r = self.conv_t_r(x_r) - self.conv_t_i(x_i)
        out_i = self.conv_t_r(x_i) + self.conv_t_i(x_r)
        return out_r, out_i

class ComplexBatchNorm2d(nn.Module):
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        super().__init__()
        self.num_features = num_features
        self.eps = eps
        self.momentum = momentum
        
        self.register_buffer('running_mean_r', torch.zeros(num_features))
        self.register_buffer('running_mean_i', torch.zeros(num_features))
        self.register_buffer('running_Vrr', torch.ones(num_features))
        self.register_buffer('running_Vii', torch.ones(num_features))
        self.register_buffer('running_Vri', torch.zeros(num_features))

        self.gamma_rr = nn.Parameter(torch.ones(num_features))
        self.gamma_ii = nn.Parameter(torch.ones(num_features))
        self.gamma_ri = nn.Parameter(torch.zeros(num_features))
        self.beta_r = nn.Parameter(torch.zeros(num_features))
        self.beta_i = nn.Parameter(torch.zeros(num_features))

    def forward(self, x_r, x_i):
        B, C, F, T = x_r.shape
        if self.training:
            mean_r = x_r.mean(dim=[0, 2, 3])
            mean_i = x_i.mean(dim=[0, 2, 3])
            self.running_mean_r.data = (1 - self.momentum) * self.running_mean_r + self.momentum * mean_r
            self.running_mean_i.data = (1 - self.momentum) * self.running_mean_i + self.momentum * mean_i
        else:
            mean_r = self.running_mean_r
            mean_i = self.running_mean_i

        x_r_c = x_r - mean_r.view(1, C, 1, 1)
        x_i_c = x_i - mean_i.view(1, C, 1, 1)

        if self.training:
            Vrr = (x_r_c ** 2).mean(dim=[0, 2, 3]) + self.eps
            Vii = (x_i_c ** 2).mean(dim=[0, 2, 3]) + self.eps
            Vri = (x_r_c * x_i_c).mean(dim=[0, 2, 3])

            self.running_Vrr.data = (1 - self.momentum) * self.running_Vrr + self.momentum * Vrr
            self.running_Vii.data = (1 - self.momentum) * self.running_Vii + self.momentum * Vii
            self.running_Vri.data = (1 - self.momentum) * self.running_Vri + self.momentum * Vri
        else:
            Vrr = self.running_Vrr
            Vii = self.running_Vii
            Vri = self.running_Vri

        det = Vrr * Vii - Vri ** 2 + self.eps
        s = torch.sqrt(det)
        t = torch.sqrt(Vrr + Vii + 2 * s + self.eps)
        inv_st = 1.0 / (t * s + self.eps)

        Wrr = ((Vii + s) * inv_st).view(1, C, 1, 1)
        Wii = ((Vrr + s) * inv_st).view(1, C, 1, 1)
        Wri = (-Vri * inv_st).view(1, C, 1, 1)

        x_r_hat = Wrr * x_r_c + Wri * x_i_c
        x_i_hat = Wri * x_r_c + Wii * x_i_c

        out_r = self.gamma_rr.view(1, C, 1, 1) * x_r_hat + self.gamma_ri.view(1, C, 1, 1) * x_i_hat + self.beta_r.view(1, C, 1, 1)
        out_i = self.gamma_ri.view(1, C, 1, 1) * x_r_hat + self.gamma_ii.view(1, C, 1, 1) * x_i_hat + self.beta_i.view(1, C, 1, 1)

        return out_r, out_i

class ComplexLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=2):
        super().__init__()
        self.lstm_r = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.lstm_i = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)

    def forward(self, x_r, x_i):
        F_rr, _ = self.lstm_r(x_r)
        F_ir, _ = self.lstm_r(x_i)
        F_ri, _ = self.lstm_i(x_r)
        F_ii, _ = self.lstm_i(x_i)

        out_r = F_rr - F_ii
        out_i = F_ri + F_ir
        return out_r, out_i

class EncoderBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = ComplexConv2d(in_c, out_c, kernel_size=(5, 2), stride=(2, 1), padding=0)
        self.bn = ComplexBatchNorm2d(out_c)
        self.prelu_r = nn.PReLU(out_c)
        self.prelu_i = nn.PReLU(out_c)

    def forward(self, x_r, x_i):
        x_r_pad = F.pad(x_r, (1, 0, 2, 2))
        x_i_pad = F.pad(x_i, (1, 0, 2, 2))
        
        x_r, x_i = self.conv(x_r_pad, x_i_pad)
        x_r, x_i = self.bn(x_r, x_i)
        x_r = self.prelu_r(x_r)
        x_i = self.prelu_i(x_i)
        return x_r, x_i

class DecoderBlock(nn.Module):
    def __init__(self, in_c, out_c, is_last=False):
        super().__init__()
        self.conv_t = ComplexConvTranspose2d(
            in_c, out_c, kernel_size=(5, 2), stride=(2, 1), 
            padding=(2, 0), output_padding=(1, 0)
        )
        self.is_last = is_last
        if not is_last:
            self.bn = ComplexBatchNorm2d(out_c)
            self.prelu_r = nn.PReLU(out_c)
            self.prelu_i = nn.PReLU(out_c)

    def forward(self, x_r, x_i):
        x_r, x_i = self.conv_t(x_r, x_i)
        
        x_r = x_r[..., :-1]
        x_i = x_i[..., :-1]
        
        if not self.is_last:
            x_r, x_i = self.bn(x_r, x_i)
            x_r = self.prelu_r(x_r)
            x_i = self.prelu_i(x_i)
        return x_r, x_i

class DCCRN(nn.Module):
    def __init__(self):
        super().__init__()
        channels = [1, 32, 64, 128, 128, 256, 256]
        
        self.encoders = nn.ModuleList()
        for i in range(6):
            self.encoders.append(EncoderBlock(channels[i], channels[i+1]))

        self.lstm = ComplexLSTM(input_size=1280, hidden_size=256, num_layers=2)
        self.dense_r = nn.Linear(256, 1280)
        self.dense_i = nn.Linear(256, 1280)

        self.decoders = nn.ModuleList()
        dec_channels = [256, 256, 128, 128, 64, 32]
        for i in range(6):
            in_c = dec_channels[i] + channels[6-i]
            out_c = channels[5-i]
            is_last = (i == 5)
            self.decoders.append(DecoderBlock(in_c, out_c, is_last))

        self.register_buffer('window', torch.hann_window(400))

    def forward(self, wav):
        stft = torch.stft(wav, n_fft=512, hop_length=100, win_length=400, 
                          window=self.window, return_complex=True)
        stft_r = stft.real.unsqueeze(1)
        stft_i = stft.imag.unsqueeze(1)

        stft_r = F.pad(stft_r, (0, 0, 0, 320 - 257))
        stft_i = F.pad(stft_i, (0, 0, 0, 320 - 257))

        enc_outputs_r, enc_outputs_i = [], []
        x_r, x_i = stft_r, stft_i
        for encoder in self.encoders:
            x_r, x_i = encoder(x_r, x_i)
            enc_outputs_r.append(x_r)
            enc_outputs_i.append(x_i)

        B, C, F_dim, T = x_r.shape  # C=256, F_dim=5
        lstm_in_r = x_r.permute(0, 3, 1, 2).reshape(B, T, C * F_dim)
        lstm_in_i = x_i.permute(0, 3, 1, 2).reshape(B, T, C * F_dim)

        lstm_out_r, lstm_out_i = self.lstm(lstm_in_r, lstm_in_i)

        dense_out_r = self.dense_r(lstm_out_r) - self.dense_i(lstm_out_i)
        dense_out_i = self.dense_r(lstm_out_i) + self.dense_i(lstm_out_r)

        x_r = dense_out_r.view(B, T, C, F_dim).permute(0, 2, 3, 1)
        x_i = dense_out_i.view(B, T, C, F_dim).permute(0, 2, 3, 1)

        for i, decoder in enumerate(self.decoders):
            skip_r = enc_outputs_r[5 - i]
            skip_i = enc_outputs_i[5 - i]
            
            x_r = torch.cat([x_r, skip_r], dim=1)
            x_i = torch.cat([x_i, skip_i], dim=1)
            
            x_r, x_i = decoder(x_r, x_i)

        out_r = x_r[:, 0, :257, :]
        out_i = x_i[:, 0, :257, :]

        mask_mag = torch.tanh(torch.sqrt(out_r**2 + out_i**2 + 1e-8))
        mask_phase = torch.atan2(out_i, out_r)
        
        mask_r = mask_mag * torch.cos(mask_phase)
        mask_i = mask_mag * torch.sin(mask_phase)

        S_r = stft.real * mask_r - stft.imag * mask_i
        S_i = stft.real * mask_i + stft.imag * mask_r
        S_complex = torch.complex(S_r, S_i)

        enh_wav = torch.istft(S_complex, n_fft=512, hop_length=100, win_length=400, 
                              window=self.window, length=wav.shape[1])
        return enh_wav


def si_snr_loss(preds, target, eps=1e-8):
    target_mean = target.mean(dim=1, keepdim=True)
    preds_mean = preds.mean(dim=1, keepdim=True)
    
    target_zm = target - target_mean
    preds_zm = preds - preds_mean

    dot_product = (target_zm * preds_zm).sum(dim=1, keepdim=True)
    target_energy = (target_zm ** 2).sum(dim=1, keepdim=True) + eps

    alpha = dot_product / target_energy
    s_target = alpha * target_zm
    e_noise = preds_zm - s_target

    s_target_energy = (s_target ** 2).sum(dim=1)
    e_noise_energy = (e_noise ** 2).sum(dim=1)

    si_snr = 10 * torch.log10(s_target_energy / (e_noise_energy + eps) + eps)
    return -si_snr.mean()

In [5]:
def clean_text(text):
    return re.sub(r'[^\w\s]', '', str(text).lower()).strip()

def get_snr_scale(signal, noise, snr_db):
    sig_p = signal.norm(p=2)**2 / (signal.numel() + 1e-8)
    noi_p = noise.norm(p=2)**2 / (noise.numel() + 1e-8)
    target_p = sig_p / (10 ** (snr_db / 10))
    return torch.sqrt(target_p / (noi_p + 1e-8))

def apply_noise(clean, force_type=None, file_seed=None):
    if file_seed is not None:
        random.seed(file_seed)
        np.random.seed(file_seed)
        torch.manual_seed(file_seed)

    snr = random.uniform(-5, 15)
    n_len = clean.shape[-1]
    allowed_noises = ['babble', 'rir', 'white']

    if force_type is not None and force_type in allowed_noises:
        noise_cat = force_type
    else:
        noise_cat = random.choice(allowed_noises)

    if noise_cat == 'babble':
        noise = BABBLE_WAVEFORM
        if noise.shape[-1] < n_len:
            repeats = (n_len // noise.shape[-1]) + 2
            noise = noise.repeat(1, repeats)
        max_start = noise.shape[-1] - n_len
        start = random.randint(0, max_start) if max_start > 0 else 0
        noise_crop = noise[:, start:start+n_len]
        scale = get_snr_scale(clean, noise_crop, snr)
        noisy = clean + noise_crop * scale
    elif noise_cat == 'rir':
        rir = RIR_WAVEFORM
        n_fft_conv = n_len + rir.shape[-1] - 1
        clean_fft = torch.fft.rfft(clean, n=n_fft_conv)
        rir_fft = torch.fft.rfft(rir, n=n_fft_conv)
        augmented = torch.fft.irfft(clean_fft * rir_fft, n=n_fft_conv)
        noisy = augmented[:, :n_len]
        white = torch.randn_like(clean)
        scale = get_snr_scale(noisy, white, snr + 10)
        noisy = noisy + white * scale
    elif noise_cat == 'white':
        noise = torch.randn(1, n_len, device=clean.device)
        noise = noise / (noise.abs().max() + 1e-8)
        scale = get_snr_scale(clean, noise, snr)
        noisy = clean + noise * scale

    max_val = noisy.abs().max()
    if max_val > 1.0:
        noisy = noisy / (max_val + 1e-8)
    return noisy

In [6]:
class WaveformDataset(Dataset):
    def __init__(self, data_dir, ref_dict, is_train=True):
        self.data_dir = data_dir
        self.ref_dict = ref_dict
        self.is_train = is_train
        file_list = []
        for r, _, fs in os.walk(data_dir):
            for f in fs:
                if f.endswith('.mp3') and f in ref_dict:
                    file_list.append(os.path.join(r, f))
        self.files = sorted(file_list)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        wav_np, sr = librosa.load(file_path, sr=None, mono=False)
        wav = torch.from_numpy(wav_np)
        if wav.ndim == 1:
            wav = wav.unsqueeze(0)

        if sr != Config.sr:
            wav = T.Resample(sr, Config.sr)(wav)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)

        if wav.shape[-1] > Config.target_samples:
            start = random.randint(0, wav.shape[-1] - Config.target_samples) if self.is_train else 0
            wav = wav[:, start:start + Config.target_samples]
        else:
            wav = F.pad(wav, (0, Config.target_samples - wav.shape[-1]))

        clean = wav
        noisy = apply_noise(clean) if self.is_train else clean
        return noisy.squeeze(0), clean.squeeze(0), self.ref_dict[os.path.basename(file_path)]


def exact_collate_fn(batch):
    noisy, clean, texts = zip(*batch)
    return torch.stack(noisy), torch.stack(clean), list(texts)

In [7]:
def si_snr_loss(preds, target, eps=1e-8):
    preds = preds.squeeze(1) if preds.ndim == 3 else preds
    target = target.squeeze(1) if target.ndim == 3 else target
    
    target_zm = target - target.mean(dim=1, keepdim=True)
    preds_zm = preds - preds.mean(dim=1, keepdim=True)
    
    dot_product = (target_zm * preds_zm).sum(dim=1, keepdim=True)
    target_energy = (target_zm ** 2).sum(dim=1, keepdim=True) + eps
    
    s_target = (dot_product / target_energy) * target_zm
    e_noise = preds_zm - s_target
    
    si_snr = 10 * torch.log10((s_target**2).sum(dim=1) / ((e_noise**2).sum(dim=1) + eps) + eps)
    return -si_snr.mean()

In [8]:
if not os.path.exists(Config.extract_dir):
        os.makedirs(Config.extract_dir, exist_ok=True)
        with tarfile.open(Config.tar_path, "r") as tar: 
            tar.extractall(path=Config.extract_dir)

df_train = pd.read_csv(Config.tsv_path, sep='\t')
reference_dict = {row['path']: clean_text(row['sentence']) for _, row in df_train.iterrows()}

import soundfile as sf
import torch

babble_path = _download_asset("tutorial-assets/Lab41-SRI-VOiCES-rm1-babb-mc01-stu-clo-8000hz.wav")
global BABBLE_WAVEFORM

waveform_np, sr_b = sf.read(babble_path, always_2d=True)
BABBLE_WAVEFORM = torch.tensor(waveform_np.T, dtype=torch.float32)
BABBLE_WAVEFORM = T.Resample(sr_b, Config.sr)(BABBLE_WAVEFORM.mean(dim=0, keepdim=True))

rir_path = _download_asset("tutorial-assets/Lab41-SRI-VOiCES-rm1-impulse-mc01-stu-clo-8000hz.wav")
global RIR_WAVEFORM

waveform_r_np, sr_r = sf.read(rir_path, always_2d=True)
RIR_WAVEFORM = torch.tensor(waveform_r_np.T, dtype=torch.float32)

RIR_WAVEFORM = T.Resample(sr_r, Config.sr)(RIR_WAVEFORM.mean(dim=0, keepdim=True))
RIR_WAVEFORM = RIR_WAVEFORM[:, :int(Config.sr * 0.3)]
RIR_WAVEFORM = RIR_WAVEFORM / torch.norm(RIR_WAVEFORM, p=2)

In [9]:
model = DCCRN().to(Config.device)
optimizer = torch.optim.Adam(model.parameters(), lr=Config.lr)

dataset = WaveformDataset(Config.extract_dir, reference_dict, is_train=True)
train_size = int(0.9 * len(dataset))
generator = torch.Generator().manual_seed(42)
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, len(dataset)-train_size], generator=generator)
train_loader = DataLoader(train_ds, batch_size=Config.batch_size, shuffle=True)

In [ ]:
for epoch in range(1, Config.epochs + 1):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}")
    
    for noisy, clean, _ in pbar:
        noisy, clean = noisy.to(Config.device), clean.to(Config.device)
        
        optimizer.zero_grad()
        enhanced = model(noisy)
        
        loss = si_snr_loss(enhanced, clean)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
        
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({"SI-SNR Loss": f"{loss.item():.4f}"})

torch.save(model.state_dict(), f"../models/dccrn_weights.pth")

Epoch 1:   0%|          | 1/5954 [00:30<49:57:23, 30.21s/it, SI-SNR Loss=27.5014]


KeyboardInterrupt: 

In [14]:
model.load_state_dict(torch.load('../models/dccrn_weights.pth', map_location=Config.device))

<All keys matched successfully>

In [15]:
seed_everything(42)

In [16]:
def evaluate_model(model, device, val_dataset, limit=20):
    asr = whisper.load_model("large-v3").to(device)
    model.eval()
    noise_types = ['babble', 'rir', 'white']
    stats = {n: {"wer_n": [], "wer_d": []} for n in noise_types}

    if limit is not None:
        indices = list(range(min(limit, len(val_dataset))))
    else:
        indices = list(range(len(val_dataset)))

    with torch.no_grad():
        for idx in tqdm(indices):
            _, clean_wav, ref_text = val_dataset[idx]
            ref_text = clean_text(ref_text)

            for n_type in noise_types:
                noisy_wav = apply_noise(clean_wav.unsqueeze(0), force_type=n_type, file_seed=idx).to(device)
                denoised_wav = model(noisy_wav)

                t_n = asr.transcribe(noisy_wav.squeeze().cpu().numpy(), fp16=False, language='ru')['text']
                t_d = asr.transcribe(denoised_wav.squeeze().cpu().numpy(), fp16=False, language='ru')['text']

                stats[n_type]["wer_n"].append(wer(ref_text, clean_text(t_n)))
                stats[n_type]["wer_d"].append(wer(ref_text, clean_text(t_d)))

    print(f"\n{'Noise Type':<10} | {'WER Noisy':<10} | {'WER Denoised':<10}")
    for n in noise_types:
        wn, wd = np.mean(stats[n]["wer_n"]), np.mean(stats[n]["wer_d"])
        print(f"{n:<10} | {wn:<10.4f} | {wd:<10.4f}")

evaluate_model(model, Config.device, val_ds, limit=20)

 51%|██████████████████▉                  | 1.47G/2.88G [02:19<02:12, 11.3MiB/s]


KeyboardInterrupt: 

In [ ]:
from jiwer import process_words
import numpy as np
import torch
import whisper
from tqdm.auto import tqdm

def evaluate_and_listen_components(model, device, val_dataset, limit=None):
    asr = whisper.load_model("large-v3").to(device)
    model.eval()
    
    noise_types = ['babble', 'rir', 'white']
    
    stats = {n: {
        "wer_n": [], "s_n": [], "d_n": [], "i_n": [],
        "wer_d": [], "s_d": [], "d_d": [], "i_d": []
    } for n in noise_types}
    
    if limit is not None:
        indices = list(range(min(limit, len(val_dataset))))
    else:
        indices = list(range(len(val_dataset)))
    
    with torch.no_grad():
        for idx in tqdm(indices, desc="Evaluation (Macro-average)"):
            _, clean_wav, ref_text = val_dataset[idx]
            clean_tensor_cpu = clean_wav.unsqueeze(0)
    
            for n_type in noise_types:
                noisy_tensor_cpu = apply_noise(clean_tensor_cpu, force_type=n_type, file_seed=idx)
                noisy_tensor = noisy_tensor_cpu.to(device)
                
                denoised_tensor = model(noisy_tensor)
    
                noisy_np = noisy_tensor.squeeze().cpu().numpy()
                denoised_np = denoised_tensor.squeeze().cpu().numpy()
    
                t_n = asr.transcribe(noisy_np, fp16=False, language='ru')['text']
                t_d = asr.transcribe(denoised_np, fp16=False, language='ru')['text']
    
                clean_ref = clean_text(ref_text)
                clean_hyp_n = clean_text(t_n)
                clean_hyp_d = clean_text(t_d)
    
                out_n = process_words(clean_ref, clean_hyp_n)
                n_words_n = out_n.substitutions + out_n.deletions + out_n.hits
                
                if n_words_n > 0:
                    stats[n_type]["wer_n"].append((out_n.substitutions + out_n.deletions + out_n.insertions) / n_words_n)
                    stats[n_type]["s_n"].append(out_n.substitutions / n_words_n)
                    stats[n_type]["d_n"].append(out_n.deletions / n_words_n)
                    stats[n_type]["i_n"].append(out_n.insertions / n_words_n)
                else:
                    stats[n_type]["wer_n"].append(0.0)
                    stats[n_type]["s_n"].append(0.0)
                    stats[n_type]["d_n"].append(0.0)
                    stats[n_type]["i_n"].append(0.0)
    
                out_d = process_words(clean_ref, clean_hyp_d)
                n_words_d = out_d.substitutions + out_d.deletions + out_d.hits
                
                if n_words_d > 0:
                    stats[n_type]["wer_d"].append((out_d.substitutions + out_d.deletions + out_d.insertions) / n_words_d)
                    stats[n_type]["s_d"].append(out_d.substitutions / n_words_d)
                    stats[n_type]["d_d"].append(out_d.deletions / n_words_d)
                    stats[n_type]["i_d"].append(out_d.insertions / n_words_d)
                else:
                    stats[n_type]["wer_d"].append(0.0)
                    stats[n_type]["s_d"].append(0.0)
                    stats[n_type]["d_d"].append(0.0)
                    stats[n_type]["i_d"].append(0.0)
    
    header = f"{'Noise':<8} | {'WER_N':<7} (S/D/I) | {'WER_D':<7} (S/D/I) | {'Gain WER':<8}"
    print(header)
    print("-" * 65)
    
    for n_type in noise_types:
        wer_n = np.mean(stats[n_type]["wer_n"])
        s_n_pct = np.mean(stats[n_type]["s_n"])
        d_n_pct = np.mean(stats[n_type]["d_n"])
        i_n_pct = np.mean(stats[n_type]["i_n"])
    
        wer_d = np.mean(stats[n_type]["wer_d"])
        s_d_pct = np.mean(stats[n_type]["s_d"])
        d_d_pct = np.mean(stats[n_type]["d_d"])
        i_d_pct = np.mean(stats[n_type]["i_d"])
    
        str_noisy = f"{wer_n:.4f} ({s_n_pct:.4f}/{d_n_pct:.4f}/{i_n_pct:.4f})"
        str_denois = f"{wer_d:.4f} ({s_d_pct:.4f}/{d_d_pct:.4f}/{i_d_pct:.4f})"
    
        print(f"{n_type:<8} | {str_noisy:<25} | {str_denois:<25} | {wer_n - wer_d:<8.4f}")

evaluate_and_listen_components(model, Config.device, val_ds, limit=20)

In [ ]:
seed_everything(42)

In [ ]:
from jiwer import process_words
import torch
import whisper
from IPython.display import Audio, display

def show_specific_examples(model, device, val_dataset, indices=[0, 1, 2]):
    asr = whisper.load_model("large-v3").to(device)
    model.eval()

    noise_types = ['babble', 'rir', 'white']

    with torch.no_grad():
        for idx in indices:
            _, clean_wav, ref_text = val_dataset[idx]
            clean_ref = clean_text(ref_text)

            print(f"ПРИМЕР (Индекс в датасете: {idx})")
            print(f"Reference : {clean_ref}")
            
            print("🔊 Оригинальное (чистое) аудио:")
            display(Audio(clean_wav.cpu().numpy(), rate=Config.sr))
            print(f"{'='*85}")

            for n_type in noise_types:
                header = f"{'Noise':<8} | {'WER_N':<7} (S/D/I) | {'WER_D':<7} (S/D/I) | {'Gain WER':<8}"
                print(header)
                print("-" * 85)
                
                clean_tensor_cpu = clean_wav.unsqueeze(0)
                
                noisy_tensor_cpu = apply_noise(clean_tensor_cpu, force_type=n_type, file_seed=idx)
                noisy_tensor = noisy_tensor_cpu.to(device)

                denoised_tensor = model(noisy_tensor)

                t_n = asr.transcribe(noisy_tensor.squeeze().cpu().numpy(), fp16=False, language='ru')['text']
                t_d = asr.transcribe(denoised_tensor.squeeze().cpu().numpy(), fp16=False, language='ru')['text']

                clean_hyp_n = clean_text(t_n)
                clean_hyp_d = clean_text(t_d)

                out_n = process_words(clean_ref, clean_hyp_n)
                n_words_n = out_n.substitutions + out_n.deletions + out_n.hits

                wer_n, s_n, d_n, i_n = 0.0, 0.0, 0.0, 0.0
                if n_words_n > 0:
                    wer_n = (out_n.substitutions + out_n.deletions + out_n.insertions) / n_words_n
                    s_n = out_n.substitutions / n_words_n
                    d_n = out_n.deletions / n_words_n
                    i_n = out_n.insertions / n_words_n

                out_d = process_words(clean_ref, clean_hyp_d)
                n_words_d = out_d.substitutions + out_d.deletions + out_d.hits

                wer_d, s_d, d_d, i_d = 0.0, 0.0, 0.0, 0.0
                if n_words_d > 0:
                    wer_d = (out_d.substitutions + out_d.deletions + out_d.insertions) / n_words_d
                    s_d = out_d.substitutions / n_words_d
                    d_d = out_d.deletions / n_words_d
                    i_d = out_d.insertions / n_words_d

                str_noisy = f"{wer_n:.4f} ({s_n:.4f}/{d_n:.4f}/{i_n:.4f})"
                str_denois = f"{wer_d:.4f} ({s_d:.4f}/{d_d:.4f}/{i_d:.4f})"

                print(f"{n_type:<8} | {str_noisy:<25} | {str_denois:<25} | {wer_n - wer_d:<8.4f}")
                print(f"  [Noisy Hyp] : {clean_hyp_n}")
                print(f"  [Denoi Hyp] : {clean_hyp_d}")
                
                print(f"\n🔊 Зашумленное аудио ({n_type}):")
                display(Audio(noisy_tensor.squeeze().cpu().numpy(), rate=Config.sr))
                
                print(f"🔊 Очищенное аудио ({n_type}):")
                display(Audio(denoised_tensor.squeeze().cpu().numpy(), rate=Config.sr))
                print("\n")

show_specific_examples(model, Config.device, val_ds, indices=[5, 12, 42])

In [ ]:
seed_everything(42)

In [ ]:
import torch.nn.functional as F
import torchaudio
import os
from IPython.display import Audio, display

def show_full_audio_examples(model, device, val_dataset, indices=[5, 12, 42]):
    asr = whisper.load_model("large-v3").to(device)
    model.eval()
    noise_types = ['babble', 'rir', 'white']

    with torch.no_grad():
        for idx in indices:
            file_path = val_dataset.dataset.files[val_dataset.indices[idx]]
            clean_wav, sr = torchaudio.load(file_path)
            
            if sr != Config.sr:
                clean_wav = torchaudio.transforms.Resample(sr, Config.sr)(clean_wav)
            if clean_wav.shape[0] > 1:
                clean_wav = clean_wav.mean(dim=0, keepdim=True)
            
            ref_text = val_dataset.dataset.ref_dict[os.path.basename(file_path)]
            clean_ref = clean_text(ref_text)

            print(f"ПРИМЕР (ПОЛНЫЙ ДАТАСЕТ): Индекс {idx}, Файл: {os.path.basename(file_path)}")
            print(f"Reference : {clean_ref}")
            print(f"Длительность: {clean_wav.shape[-1]/Config.sr:.2f} сек.")
            
            print("Чистое аудио:")
            display(Audio(clean_wav.squeeze().numpy(), rate=Config.sr))
            print(f"{'='*95}")

            for n_type in noise_types:
                header = f"{'Noise':<8} | {'WER_N':<7} (S/D/I) | {'WER_D':<7} (S/D/I) | {'Gain WER':<8}"
                print(header)

                noisy_tensor_cpu = apply_noise(clean_wav, force_type=n_type, file_seed=idx)
                noisy_tensor = noisy_tensor_cpu.to(device)

                denoised_tensor = model(noisy_tensor)

                noisy_np = noisy_tensor.squeeze().cpu().numpy()
                denoised_np = denoised_tensor.squeeze().cpu().numpy()
                
                t_n = asr.transcribe(noisy_np, fp16=False, language='ru')['text']
                t_d = asr.transcribe(denoised_np, fp16=False, language='ru')['text']

                clean_hyp_n = clean_text(t_n)
                clean_hyp_d = clean_text(t_d)

                out_n = process_words(clean_ref, clean_hyp_n)
                out_d = process_words(clean_ref, clean_hyp_d)
                
                n_words = out_n.substitutions + out_n.deletions + out_n.hits
                
                def get_stats(out, nw):
                    if nw == 0: return 0.0, 0.0, 0.0, 0.0
                    wer = (out.substitutions + out.deletions + out.insertions) / nw
                    return wer, out.substitutions/nw, out.deletions/nw, out.insertions/nw

                wer_n, s_n, d_n, i_n = get_stats(out_n, n_words)
                wer_d, s_d, d_d, i_d = get_stats(out_d, n_words)

                str_noisy = f"{wer_n:.4f} ({s_n:.2f}/{d_n:.2f}/{i_n:.2f})"
                str_denois = f"{wer_d:.4f} ({s_d:.2f}/{d_d:.2f}/{i_d:.2f})"

                print(f"{n_type:<8} | {str_noisy:<25} | {str_denois:<25} | {wer_n - wer_d:<8.4f}")
                print(f"  [Noisy Hyp] : {clean_hyp_n}")
                print(f"  [Denoi Hyp] : {clean_hyp_d}")
                
                print(f"Зашумленное ({n_type}):")
                display(Audio(noisy_np, rate=Config.sr))
                print(f"Очищенное ({n_type}):")
                display(Audio(denoised_np, rate=Config.sr))
                print("-" * 40)

show_full_audio_examples(model, Config.device, val_ds, indices=[5, 12, 42])